# GraphDB Database

[GraphDB](https://graphdb.ontotext.com/) is an RDF graph database (triplestore) developed by Ontotext, with a SPARQL 1.1 endpoint for each repository. The Free edition runs without a license.

CIMantic Graphs provides the `GraphDBConnection` class for reading and updating CIM models in a GraphDB repository. Like [Blazegraph](3_3_blazegraph.ipynb), it is a `SPARQLEndpointConnection`, so it has the same methods and queries.

----

## Installation and Setup

### Python Dependencies

`GraphDBConnection` needs the `SPARQLWrapper` library, from the `sparql` extra:

```bash
pip install "cim-graph[sparql]"
```

### Local Instance (docker compose)

The repository's `docker-compose.yml` runs GraphDB 10.8 Free. From a CIM-Graph checkout:

```bash
docker compose up -d --wait graphdb
uv run python tests/seed_databases.py graphdb   # loads tests/test_models/ieee13.xml
```

The seed script creates the repository `powergrid_models` with the `empty` ruleset (no inference), so queries see only the triples in the file. Its SPARQL endpoint is `http://localhost:7200/repositories/powergrid_models`, and the GraphDB Workbench is at http://localhost:7200. The container has no volume, so `docker compose down` discards the data.

----

## Environment Configuration

A connection reads its settings from `CIMG_*` environment variables when it is created (see [Environment Variables](3_2_env_variables.ipynb)). `CIMG_URL` is the repository endpoint, `http://<host>:7200/repositories/<repository>`. For the docker compose instance:

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
os.environ['CIMG_URL'] = 'http://localhost:7200/repositories/powergrid_models'
os.environ['CIMG_IEC61970_552'] = '552-NEW'

import cimprofiles.cim17.cimhub_2023 as cim

----

## Creating a Connection

In [2]:
from cimgraph.databases import GraphDBConnection

db = GraphDBConnection()

The SPARQLWrapper client is created on the first query. Queries are sent by POST to `CIMG_URL`, and SPARQL updates to `CIMG_URL/statements`.

----

## Usage Examples

### Example 1: Retrieving an Object by mRID

In [3]:
feeder = db.get_object(mRID="49AD8E07-3BF9-A4E2-CB8F-C3722F837B62")
feeder.pprint()

{
    "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
    "@type": "Feeder"
}


### Example 2: Loading a Feeder Model

In [4]:
from cimgraph.models import FeederModel

network = FeederModel(container=feeder, connection=db)
print(f"Loaded {len(network.graph[cim.ACLineSegment])} line segments")

Loaded 11 line segments


### Example 3: Expanding the Graph

`get_all_edges()` adds the attributes and associated objects of every object of a class. Unit values are `CIMUnit` objects in SI base units; convert them with `.to()`:

In [5]:
network.get_all_edges(cim.ACLineSegment)

line = network.first(cim.ACLineSegment)
print(f"Line {line.name}: length {line.length.to('m')} m ({line.length.to('ft'):.0f} ft)")

Line 645646: length 91.44 m (300 ft)


### Example 4: Querying Object Attributes

In [6]:
name = db.get_from_triple(subject=feeder, predicate='IdentifiedObject.name')
print(f"Feeder name: {name}")

substation = db.get_from_triple(subject=feeder, predicate='Feeder.NormalEnergizingSubstation')
print(f"Substation: {substation}")

Feeder name: ['ieee13nodeckt']
Substation: [{"@id": "6c62c905-6fc7-653d-9f1e-1340f974a587", "@type": "Substation"}]


### Example 5: Executing a Custom SPARQL Query

`execute()` returns the SPARQL JSON results as a `dict`:

In [7]:
query_text = '''
PREFIX r:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX cim:  <http://iec.ch/TC57/CIM100#>
SELECT ?name (COUNT(?line) AS ?lines)
WHERE {
    ?line r:type cim:ACLineSegment ;
          cim:Equipment.EquipmentContainer ?feeder .
    ?feeder cim:IdentifiedObject.name ?name .
}
GROUP BY ?name
'''

for result in db.execute(query_text)['results']['bindings']:
    print(f"{result['name']['value']}: {result['lines']['value']} line segments")

ieee13nodeckt: 11 line segments


The [Blazegraph page](3_3_blazegraph.ipynb) describes the rest of the `SPARQLEndpointConnection` methods, the query batching and the sequence diagrams, which apply to GraphDB as well.

----

## References

* [GraphDB Documentation](https://graphdb.ontotext.com/documentation/10.8/)
* [SPARQL 1.1 Specification](https://www.w3.org/TR/sparql11-query/)
* [Common SPARQL Methods](3_1_databases_overview.ipynb#common-sparql-endpoint-methods)
* [FeederModel Usage](../04_graph_models/4_2_feeder_model.ipynb)